# Planner Agent, Parallel Research & Report Agent Test Suite

Tests Supervisor routing across simple routes (Direct, RAG, Web, SQL) and complex multi-step research requests (Planner -> Parallel Research -> Critic -> Report Agent).

In [ ]:
import sys
from pathlib import Path

# Resolve project root
cwd = Path.cwd().resolve()
if cwd.name == "tests":
    project_root = cwd.parent.parent
elif cwd.name == "backend":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.app.agents.graph.workflow import create_workflow
from backend.app.agents.graph.nodes.planner import planner_node
from backend.app.agents.graph.nodes.researcher import parallel_research_node
from backend.app.llm.groq_provider import GroqProvider
from backend.app.llm.llm_client import LLMClient

provider = GroqProvider()
llm_client = LLMClient(provider)
workflow = create_workflow(llm_client)
print("Multi-Agent Research Workflow compiled successfully!")

In [ ]:
# Test 1 — Simple Direct Route
r1 = await workflow.ainvoke({"user_message": "What is Python?"})
print("Test 1 (Direct):", r1["route"])
assert r1["route"] == "direct"

In [ ]:
# Test 2 — RAG Route
r2 = await workflow.ainvoke({"user_message": "How many annual leave days does NexaTech provide?"})
print("Test 2 (RAG):", r2["route"])
assert r2["route"] == "rag"

In [ ]:
# Test 3 — SQL Route
r3 = await workflow.ainvoke({"user_message": "What is the total revenue in the sales database?"})
print("Test 3 (SQL):", r3["route"])
assert r3["route"] == "sql"

In [ ]:
# Test 4 — Web Route
r4 = await workflow.ainvoke({"user_message": "What are the latest developments in agentic AI?"})
print("Test 4 (Web):", r4["route"])
assert r4["route"] == "web"

In [ ]:
# Test 5 — Complex Research Request (SQL + Web)
q5 = "Analyze Q1 sales and compare the results with recent agentic AI industry developments."
r5 = await workflow.ainvoke({"user_message": q5})
print("Test 5 (Research Route):", r5["route"])
print("Research Plan:", r5.get("research_plan"))
print("Final Report Preview:", r5["final_response"][:300])
assert r5["route"] == "research"
assert "research_plan" in r5
assert "research_results" in r5
assert len(r5["final_response"]) > 0

In [ ]:
# Test 6 — RAG + Web Research Request
q6 = "Compare our remote-work policy with recent enterprise workplace trends."
r6 = await workflow.ainvoke({"user_message": q6})
print("Test 6 (RAG + Web Research Route):", r6["route"])
print("Research Plan:", r6.get("research_plan"))
assert r6["route"] == "research"

In [ ]:
# Test 7 — Parallel Execution Verification
plan_state = {
    "user_message": "Test parallel research",
    "research_plan": [
        {"task_id": "task_1", "task": "Select total sales count", "source": "sql"},
        {"task_id": "task_2", "task": "Search AI trends", "source": "web"},
        {"task_id": "task_3", "task": "Retrieve leave policy", "source": "rag"},
    ]
}
p_res = await parallel_research_node(plan_state, llm_client)
print("Parallel Execution Results Count:", len(p_res["research_results"]))
assert len(p_res["research_results"]) == 3
sources_found = {item["source"] for item in p_res["research_results"]}
assert sources_found == {"sql", "web", "rag"}

In [ ]:
# Test 8 & 9 — Security & Source Metadata Verification
sample_report = r5["final_response"]
assert "# " in sample_report or "## " in sample_report
print("Source Metadata Verified in Final Report!")